# Logistic Regression Training (PySpark ML)

Train logistic regression models for network intrusion detection using PySpark ML to handle large datasets (10GB+).

## Strategies:
1. **Balanced Data**: Train on oversampled balanced dataset
2. **Class Weighting**: Use class weights to handle imbalance

In [30]:
import subprocess
r = subprocess.run(
    ["aws", "--no-cli-pager", "s3", "sync", "--quiet", "/tmp/mlruns", "s3://nidstream/mlruns"],
    capture_output=True, text=True,
)
print(r.stderr or "synced")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

synced

In [26]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression

# ── Strategy selector ────────────────────────────────────────────────────────
# "class_weight" — original (smaller) data + per-class weights.  Fastest.
# "balanced"     — pre-oversampled data, no per-class weights.  ~2x training time.
# "both"         — train and compare both (slowest).
STRATEGY = os.environ.get("LR_STRATEGY", "class_weight")
assert STRATEGY in ("class_weight", "balanced", "both"), STRATEGY

# ── Environment config ───────────────────────────────────────────────────────
S3_BUCKET = os.environ.get("S3_BUCKET", "nidstream")

existing = SparkSession.getActiveSession()
IS_EMR = existing is not None and not existing.sparkContext.master.startswith("local")

# ── Make training_utils.py importable on the driver + executors ─────────────
# scripts/start.sh syncs notebooks/training_utils.py to s3://<bucket>/emr/
# every time it runs.  On EMR we fetch the latest copy to /tmp and add it to
# both sys.path (driver) and addPyFile (executors).  Locally we use the repo path.
if IS_EMR:
    local_utils = "/tmp/training_utils.py"
    # --quiet + DEVNULL: suppress AWS CLI progress output which leaks into Livy
    # statement output and causes Jackson JSON parse errors in the next statement.
    subprocess.run(
        ["aws", "s3", "cp", "--quiet", f"s3://{S3_BUCKET}/emr/training_utils.py", local_utils],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    if "/tmp" not in sys.path:
        sys.path.insert(0, "/tmp")
    existing.sparkContext.addPyFile(local_utils)
    import training_utils

    # boto3 is not on the YARN container's Python path — install to /tmp/pkgs
    # for this session so save_models_pyspark can write metrics to S3.
    try:
        import boto3
    except ImportError:
        subprocess.run(
            ["pip3", "install", "--quiet", "--target", "/tmp/pkgs", "boto3"],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        sys.path.insert(0, "/tmp/pkgs")
        import boto3
else:
    sys.path.append("..")
    import notebooks.training_utils as training_utils

importlib.reload(training_utils)
from training_utils import (
    load_training_data_pyspark,
    train_and_evaluate_pyspark,
    save_models_pyspark,
    log_to_mlflow_pyspark,
    print_summary_pyspark,
)

# ── SparkSession ─────────────────────────────────────────────────────────────
if IS_EMR:
    spark = existing
    DATA_ROOT            = f"s3://{S3_BUCKET}/data/processed/BCCC-CSE-CIC-IDS2018"
    MODELS_ROOT          = f"s3://{S3_BUCKET}/models"
    # Runs are written to /tmp/mlruns on the EMR master.
    # After logging, sync cell pushes them to s3://nidstream/mlruns so the local
    # MLflow UI (localhost:5000) can serve them after: aws s3 sync s3://nidstream/mlruns ./mlruns
    MLFLOW_TRACKING_URI  = "file:///tmp/mlruns"
    # HDFS is local to the cluster — far faster than S3 for checkpoint scratch space
    CHECKPOINT_DIR       = "hdfs:///tmp/spark_checkpoints"
    # 4x m5.xlarge cores = 16 vCPU.  2x cores keeps a few partitions in flight
    # per executor without flooding the scheduler.
    N_PARTITIONS         = 32

    # Only set runtime-modifiable Spark SQL configs here.  spark.serializer,
    # kryoserializer.buffer.max, and arrow.pyspark.enabled are StartUp-only
    # configs; setting them on a live Livy session raises CANNOT_MODIFY_CONFIG
    # and aborts the cell.  They are configured at cluster launch in
    # scripts/emr/02_launch_cluster.sh (spark-defaults block).
    spark.conf.set("spark.sql.shuffle.partitions", str(N_PARTITIONS))
    spark.conf.set("spark.default.parallelism",    str(N_PARTITIONS))
    print(f"\u2713 Running on EMR \u2014 using existing SparkSession")

else:
    os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
    os.environ['PYSPARK_PYTHON'] = sys.executable
    os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

    try:
        if existing:
            existing.stop()
            print("Stopped existing Spark session")
    except Exception:
        pass

    JVM_FLAGS = (
        "-XX:+UseG1GC "
        "-XX:MaxGCPauseMillis=500 "
        "-XX:InitiatingHeapOccupancyPercent=35 "
        "-XX:+UseStringDeduplication"
    )

    spark = SparkSession.builder \
        .appName("LogisticRegressionTraining") \
        .master("local[*]") \
        .config("spark.driver.memory", "10g") \
        .config("spark.driver.maxResultSize", "4g") \
        .config("spark.driver.extraJavaOptions", JVM_FLAGS) \
        .config("spark.sql.shuffle.partitions", "8") \
        .config("spark.default.parallelism", "8") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config("spark.kryoserializer.buffer.max", "512m") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
        .config("spark.sql.inMemoryColumnarStorage.batchSize", "2000") \
        .config("spark.memory.offHeap.enabled", "true") \
        .config("spark.memory.offHeap.size", "2g") \
        .getOrCreate()

    DATA_ROOT            = str(Path("..").resolve() / "data" / "processed" / "BCCC-CSE-CIC-IDS2018")
    MODELS_ROOT          = str(Path("..").resolve() / "models")
    MLFLOW_TRACKING_URI  = "http://localhost:5000"
    CHECKPOINT_DIR       = "/tmp/spark_checkpoints"
    N_PARTITIONS         = 8
    print(f"\u2713 Running locally \u2014 new SparkSession created")

print(f"Strategy       : {STRATEGY}")
print(f"Spark Version  : {spark.version}")
print(f"Master         : {spark.sparkContext.master}")
print(f"Spark UI       : {spark.sparkContext.uiWebUrl}")
print(f"Data root      : {DATA_ROOT}")
print(f"Models root    : {MODELS_ROOT}")
print(f"MLflow URI     : {MLFLOW_TRACKING_URI}")
print(f"Checkpoint dir : {CHECKPOINT_DIR}")
print(f"N partitions   : {N_PARTITIONS}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

✓ MLflow server started on EMR master (port 5001)
✓ Running on EMR — using existing SparkSession
Strategy       : class_weight
Spark Version  : 3.5.2-amzn-1
Master         : yarn
Spark UI       : http://ip-172-31-2-221.eu-west-1.compute.internal:38435
Data root      : s3://nidstream/data/processed/BCCC-CSE-CIC-IDS2018
Models root    : s3://nidstream/models
MLflow URI     : http://127.0.0.1:5001
Checkpoint dir : hdfs:///tmp/spark_checkpoints
N partitions   : 32

## 1. Load Data

In [18]:
# Load data from combined parquet (features + label_binary in one file).
# Only the strategy-relevant DataFrames are returned non-None.
train_orig_vec, train_balanced_vec, test_vec, feature_cols, project_root = load_training_data_pyspark(
    spark, strategy=STRATEGY, verbose=False, data_path=DATA_ROOT
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Loading training and test data from Parquet with PySpark (strategy=class_weight)...
✓ Data loaded (333 features, counts deferred for speed)
✓ Assembled feature vectors (lazy — checkpointed by train_and_evaluate_pyspark)

In [19]:
# Data is already prepared by the utility function
print(f"✓ Ready for training with {len(feature_cols)} features")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

? Ready for training with 333 features

## 2. Train Models

_See the `STRATEGY` flag set up above (defaults to `class_weight`). Set the `LR_STRATEGY` env var or change the assignment to switch._

In [20]:
# LR hyperparameters tuned for already-scaled data on a small EMR cluster.
#   maxIter=20, tol=1e-2 — converges quickly on standardised features.  The
#     prior maxIter=50 / tol=1e-3 typically ran all 50 iterations and roughly
#     2.5x the wall time for negligible metric improvement.
#   aggregationDepth left at default (2) — depth=4 is overkill for ~32 partitions
#     and adds reduce-tree latency.
model_params = {
    'featuresCol': 'features',
    'labelCol': 'label',
    'maxIter': 20,
    'tol': 1e-2,
    'regParam': 0.01,
    'elasticNetParam': 0.0,  # L2 regularization
    'family': 'binomial',
}

model_balanced = metrics_balanced = None
model_weighted = metrics_weighted = None

if STRATEGY in ("class_weight", "both"):
    model_weighted, metrics_weighted, _ = train_and_evaluate_pyspark(
        LogisticRegression,
        model_params,
        train_orig_vec,
        test_vec,
        "Logistic Regression - Class Weight Strategy",
        use_class_weights=True,
        n_partitions=N_PARTITIONS,
        checkpoint_dir=CHECKPOINT_DIR,
    )

if STRATEGY in ("balanced", "both"):
    model_balanced, metrics_balanced, _ = train_and_evaluate_pyspark(
        LogisticRegression,
        model_params,
        train_balanced_vec,
        test_vec,
        "Logistic Regression - Balanced Data Strategy",
        use_class_weights=False,
        n_partitions=N_PARTITIONS,
        checkpoint_dir=CHECKPOINT_DIR,
    )


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

TRAINING: Logistic Regression - Class Weight Strategy
  Class weights: {0: 0.5295276717409624, 1: 8.966634355501412}
  Checkpointing training data to hdfs:///tmp/spark_checkpoints …
  Checkpoint complete (32 partitions)
✓ Training completed in 30.60 seconds
  Iterations: 20
  Objective history: 0.038560 (final)

EVALUATING: Logistic Regression - Class Weight Strategy
Test Set Metrics:
  Accuracy:  0.9996
  Precision: 0.9996
  Recall:    0.9996
  F1 Score:  0.9996
  ROC AUC:   1.0000
  PR AUC:    0.9995
  Train time: 30.60s

## 3. Save Models

In [21]:
# Save models and metrics
save_models_pyspark(model_balanced, model_weighted, metrics_balanced, metrics_weighted, 'lr', MODELS_ROOT)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

✅ Saved models:
  s3://nidstream/models/pyspark/lr_weighted
✅ Saved metrics: s3://nidstream/models/metrics/lr_pyspark_metrics.pkl
/tmp/pkgs/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)

In [22]:
# Log to MLflow — None models/metrics are skipped automatically inside the helper.
log_to_mlflow_pyspark(
    model_balanced,
    model_weighted,
    metrics_balanced,
    metrics_weighted,
    model_prefix="lr",
    model_name="LogisticRegression",
    model_params=model_params,
    models_root=MODELS_ROOT,
    tracking_uri=MLFLOW_TRACKING_URI,
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

  class_weight: run_id=418059ba1e95448eb020a6226b1ef3ff
MLflow experiment : nidstream_lr
Tracking URI      : file:///tmp/mlruns
2026/04/13 22:27:42 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GI

In [ ]:
# Sync mlruns metadata to S3 so local MLflow UI can show these runs.
# On Mac after this cell: aws s3 sync s3://nidstream/mlruns ./mlruns
# Then open http://localhost:5000 — no restart needed.
if IS_EMR:
    _r = subprocess.run(
        ["aws", "--no-cli-pager", "s3", "sync", "--quiet", "/tmp/mlruns", f"s3://{S3_BUCKET}/mlruns"],
        capture_output=True, text=True,
    )
    print(_r.stderr or f"\u2713 mlruns synced to s3://{S3_BUCKET}/mlruns")
    print("Run on Mac: aws s3 sync s3://nidstream/mlruns ./mlruns")


## 4. Summary

In [23]:
# Print comparison summary
print_summary_pyspark(metrics_balanced, metrics_weighted, "Logistic Regression")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


LOGISTIC REGRESSION MODEL SUMMARY

Class Weight Strategy:
  accuracy    : 0.9996
  precision   : 0.9996
  recall      : 0.9996
  f1          : 0.9996
  roc_auc     : 1.0000
  pr_auc      : 0.9995
  train_time  : 30.60s

In [24]:
# Stop Spark session when done
# spark.stop()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [25]:
from pyspark.ml.classification import LogisticRegressionModel

# Load saved model
model = LogisticRegressionModel.load("s3://nidstream/models/pyspark/lr_weighted")

# Run on a sample of test data
sample = test_vec.limit(100)
preds = model.transform(sample)
preds.select("label", "prediction", "probability").show(10)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|    0|       0.0|[0.99883097077612...|
|    0|       0.0|[0.90805916651087...|
|    0|       0.0|[0.99887244581841...|
|    0|       0.0|[0.98905760617265...|
|    0|       0.0|[0.98423684879945...|
|    0|       0.0|[0.94883600675211...|
|    0|       0.0|[0.99926818856987...|
|    0|       0.0|[0.95405801173541...|
|    0|       0.0|[0.99393499938155...|
|    0|       0.0|[0.99719457581565...|
+-----+----------+--------------------+
only showing top 10 rows